# Formats: E2M1, E4M3 and NVFP4 into the bits of an f32

How the decoders build each f32 directly from its bits: the sign moves to bit 31, the exponent is re-biased (E' = e + 126 for E2M1, e + 120 for E4M3) and the mantissa is shifted to the top of the 23 mantissa bits.
An NVFP4 value is one E2M1 code (two per byte, low nibble first) times its E4M3 block scale (one per 16 values) times the F32 scale of the tensor.

In [ ]:
import re
import subprocess
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# the notebook sits in Claude Benchmark/<name>/, the repository two levels up
HERE = Path.cwd().resolve()
ROOT = HERE.parents[1]

MM = 1 / 25.4
INK, PAPER, GREY, SHADE = "#1f1d1a", "#faf8f3", "#77746e", "#d3cfc6"
BASELINE_GREY = "#a3a09a"
plt.rcParams.update({
    "font.family": "Liberation Sans", "font.size": 7, "axes.labelsize": 7,
    "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6.5,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.spines.top": False, "axes.spines.right": False,
})


def main_constant(name):
    """A constant of src/main.zig, such as `const MIN_BLOCKS_FOR_WORKERS = 16 * 1024;`, evaluated."""
    source = (ROOT / "src" / "main.zig").read_text(encoding="utf-8")
    expression = re.search(rf"^const {name} = ([0-9 *]+);", source, re.MULTILINE).group(1)
    return int(np.prod([int(factor) for factor in expression.split("*")]))


def run_benchmark(bench, csv):
    """Runs the benchmark (calls of bufferToWorker only) when its results are missing."""
    if not (HERE / csv).exists():
        subprocess.run(["zig", "run", "-OReleaseFast", "-mcpu=native", "--dep", "dequantizer",
                        f"-Mroot={HERE / bench}", f"-Mdequantizer={ROOT / 'src' / 'root.zig'}"],
                       cwd=ROOT, check=True)
    return pd.read_csv(HERE / csv)


In [ ]:
from matplotlib.colors import to_rgb
from matplotlib.patches import FancyArrowPatch, Rectangle

plt.rcParams.update({"mathtext.fontset": "custom", "mathtext.rm": "Liberation Sans",
                     "mathtext.it": "Liberation Sans:italic", "mathtext.bf": "Liberation Sans:bold"})
MONO = "Liberation Mono"

# one colour per field: brick for the sign, ink blue for the exponent, ochre for the mantissa
FIELD = {"s": "#b5533c", "e": "#34558b", "m": "#d1a042"}
# letters and arrows: the ochre is too light to read on its own tint, so it is darkened
FIELD_INK = {"s": "#b5533c", "e": "#34558b", "m": "#8a6416"}
LINE = "#c9c3b8"

def tint(colour, amount):
    paper = np.array(to_rgb(PAPER))
    return tuple(paper + amount * (np.array(to_rgb(colour)) - paper))

fig, ax = plt.subplots(figsize=(175 * MM, 152 * MM))
ax.set_xlim(0, 17.5)
ax.set_ylim(2.4, 17.5)
ax.set_aspect("equal")
ax.axis("off")

def source_bits(x, y, fields, cell=0.6, height=0.6, letters=True, top_bit=None):
    """A small format, fields = [(kind, count)] from the top bit down; returns the x span of each field."""
    spans, cursor = [], x
    for kind, count in fields:
        start = cursor
        for _ in range(count):
            ax.add_patch(Rectangle((cursor, y), cell, height, facecolor=tint(FIELD[kind], 0.25), edgecolor=PAPER, lw=0.8))
            if letters:
                ax.text(cursor + cell / 2, y + height / 2, kind, ha="center", va="center", fontsize=7.5,
                        color=FIELD_INK[kind], style="italic")
            cursor += cell
        spans.append((kind, start, cursor))
    ax.add_patch(Rectangle((x, y), cursor - x, height, fill=False, edgecolor=INK, lw=0.7))
    if top_bit is not None:
        ax.text(x + cell / 2, y + height + 0.1, str(top_bit), ha="center", va="bottom", fontsize=5.5, color=GREY)
        ax.text(cursor - cell / 2, y + height + 0.1, "0", ha="center", va="bottom", fontsize=5.5, color=GREY)
    return spans

F32_X, F32_CELL, F32_HEIGHT = 0.5, 0.33, 0.42

def f32_bits(y, styles, x=F32_X, cell=F32_CELL, height=F32_HEIGHT, labels=True):
    """The 32 bits of an f32, bit 31 on the left; styles[i] = (kind, strength), kind None for a bit left at 0."""
    for position, (kind, strength) in enumerate(styles):
        zero = kind is None
        ax.add_patch(Rectangle((x + position * cell, y), cell, height,
                               facecolor=PAPER if zero else tint(FIELD[kind], strength),
                               edgecolor=LINE if zero else PAPER, lw=0.5 if zero else 0.6))
    ax.add_patch(Rectangle((x, y), 32 * cell, height, fill=False, edgecolor=INK, lw=0.7))
    if labels:
        for position, label in ((0.0, "31"), (4.0, "30 – 23"), (20.0, "22 – 0")):
            ax.text(x + position * cell + cell / 2, y - 0.12, label, ha="center", va="top", fontsize=5.5, color=GREY)
        ax.text(x + 32 * cell + 0.2, y + height / 2, "f32", fontsize=7, color=GREY, va="center")

def centre(first, last):
    """x of the middle of f32 bits first..last (bit numbers, 31 on the left)."""
    return F32_X + ((31 - first) + (31 - last) + 1) / 2 * F32_CELL

def arrow(start, end, colour):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=8, color=colour, lw=0.8,
                                 shrinkA=2, shrinkB=2))

def notes(x, y, value):
    """What a format means, as a formula."""
    ax.text(x, y, value, fontsize=7, color=INK, va="center")

def into_f32(y, name, exponent_bits, mantissa_bits):
    """A small float above the f32 it becomes, one arrow per field; returns the y of the f32 row.
    The source cells are sized so its exponent | mantissa boundary sits right above bits 23 | 22 of the f32."""
    ax.text(0.5, y + 0.95, name, fontsize=9, fontweight="bold", color=INK, va="bottom")
    cell = 9 * F32_CELL / (1 + exponent_bits)
    spans = source_bits(F32_X, y, [("s", 1), ("e", exponent_bits), ("m", mantissa_bits)], cell,
                        top_bit=exponent_bits + mantissa_bits)
    target_y = y - 1.1
    f32_bits(target_y, [("s", 0.25)] + [("e", 0.25)] * 8 + [("m", 0.25)] * mantissa_bits
             + [(None, 0)] * (23 - mantissa_bits))
    # the dashed guide shows the alignment: the exponent ends right where it ends in the f32
    boundary = F32_X + 9 * F32_CELL
    ax.plot([boundary, boundary], [target_y + F32_HEIGHT, y], color=GREY, lw=0.5, ls=(0, (2, 2)))
    # sign to bit 31, re-biased exponent to bits 30-23, mantissa to the top of bits 22-0 (normals only)
    targets = {"s": centre(31, 31), "e": centre(30, 23), "m": centre(22, 23 - mantissa_bits)}
    for kind, start, end in spans:
        arrow(((start + end) / 2, y), (targets[kind], target_y + F32_HEIGHT), FIELD_INK[kind])
    return target_y

# ---------- F32: what every format below becomes, bias 127 ----------
y = 15.8
ax.text(0.5, y + 0.62, "F32", fontsize=9, fontweight="bold", color=INK, va="bottom")
f32_bits(y, [("s", 0.25)] + [("e", 0.25)] * 8 + [("m", 0.25)] * 23)
# the names the formulas below use for each field of the f32
for label, kind, first, last in ((r"$s$", "s", 31, 31), (r"$E'$", "e", 30, 23), (r"$M'$", "m", 22, 0)):
    ax.text(centre(first, last), y + F32_HEIGHT / 2, label, ha="center", va="center", fontsize=7.5, color=FIELD_INK[kind])
notes(0.5, y - 0.75, r"value $= (-1)^s \cdot 2^{E' - 127} \cdot (1 + M' / 2^{23})$")

# ---------- E2M1: 4 bits, bias 1 ----------
target_y = into_f32(12.9, "E2M1", 2, 1)
notes(0.5, target_y - 0.85,
      r"value $= (-1)^s \cdot 2^{e - 1} \cdot (1 + m/2)$,   and for $e = 0$ (subnormal): $(-1)^s \cdot m/2$")

# ---------- E4M3: 8 bits, bias 7 ----------
target_y = into_f32(8.6, "E4M3", 4, 3)
notes(0.5, target_y - 0.85,
      r"value $= (-1)^s \cdot 2^{e - 7} \cdot (1 + m/8)$,   and for $e = 0$ (subnormal): $(-1)^s \cdot 2^{-6} \cdot m/8$")

# ---------- NVFP4: two E2M1 codes per byte, each scaled by its block and its tensor ----------
y, cell, height = 3.95, 0.36, 0.5
ax.text(0.5, y + 1.3, "NVFP4", fontsize=9, fontweight="bold", color=INK, va="bottom")
nibble = [("s", 1), ("e", 2), ("m", 1)]
# one byte, high nibble on the left: the first value is in the low nibble
source_bits(0.5, y, nibble, cell, height)
source_bits(0.5 + 4 * cell, y, nibble, cell, height)
ax.text(0.5 + cell / 2, y + height + 0.1, "7", ha="center", va="bottom", fontsize=5.5, color=GREY)
ax.text(0.5 + 7.5 * cell, y + height + 0.1, "0", ha="center", va="bottom", fontsize=5.5, color=GREY)
ax.text(0.5 + 2 * cell, y + 0.9, "second", ha="center", va="bottom", fontsize=5.8, color=GREY)
ax.text(0.5 + 6 * cell, y + 0.9, "first", ha="center", va="bottom", fontsize=5.8, color=GREY)
ax.text(0.5 + 2 * cell, y - 0.12, "high nibble", ha="center", va="top", fontsize=5.8, color=GREY)
ax.text(0.5 + 6 * cell, y - 0.12, "low nibble", ha="center", va="top", fontsize=5.8, color=GREY)
arrow((3.55, y + height / 2), (4.35, y + height / 2), GREY)

# each code on its own: code * (scale of its block * scale of the tensor) = one f32
source_bits(4.5, y, nibble, cell, height)
ax.text(4.5 + 2 * cell, y - 0.12, "one E2M1 code", ha="center", va="top", fontsize=5.8, color=GREY)
ax.text(6.3, y + height / 2, r"$\times$", ha="center", va="center", fontsize=11, color=INK)
ax.text(6.65, y + height / 2, "(", ha="center", va="center", fontsize=14, color=INK)
source_bits(6.85, y, [("s", 1), ("e", 4), ("m", 3)], cell, height)
ax.text(6.85 + 4 * cell, y - 0.12, "E4M3 scale, one per 16 codes", ha="center", va="top", fontsize=5.8, color=GREY)
ax.text(10.1, y + height / 2, r"$\times$", ha="center", va="center", fontsize=11, color=INK)
f32_bits(y, [("s", 0.25)] + [("e", 0.25)] * 8 + [("m", 0.25)] * 23, x=10.45, cell=0.09, height=height, labels=False)
ax.text(10.45 + 32 * 0.09 / 2, y - 0.12, "f32 scale2, one per tensor", ha="center", va="top", fontsize=5.8, color=GREY)
ax.text(13.55, y + height / 2, ")", ha="center", va="center", fontsize=14, color=INK)
ax.text(13.95, y + height / 2, "=", ha="center", va="center", fontsize=11, color=INK)
f32_bits(y, [("s", 0.25)] + [("e", 0.25)] * 8 + [("m", 0.25)] * 23, x=14.3, cell=0.09, height=height, labels=False)
ax.text(14.3 + 32 * 0.09 / 2, y - 0.12, "one f32 per code", ha="center", va="top", fontsize=5.8, color=GREY)

notes(0.5, y - 0.85,
      r"value $=$ code $\times$ (scale $\times$ scale2): the two scales are multiplied once per block"
      r", and that product is shared by its 16 codes")

# which colour is which field, and the bits left at 0
x = 7.6
for kind, label in (("s", "sign"), ("e", "exponent"), ("m", "mantissa"), (None, "bit left at 0")):
    if kind is None:
        ax.add_patch(Rectangle((x, 17.0), 0.32, 0.32, facecolor=PAPER, edgecolor=GREY, lw=0.8))
    else:
        ax.add_patch(Rectangle((x, 17.0), 0.32, 0.32, facecolor=tint(FIELD[kind], 0.25), edgecolor=FIELD[kind], lw=0.8))
    ax.text(x + 0.45, 17.16, label, fontsize=7, color=INK, va="center")
    x += 0.45 + 0.17 * len(label) + 0.8

fig.savefig(HERE / "formats.png", dpi=300, bbox_inches="tight", pad_inches=0.2)